#### RFE and Mutual Information to choose optimal variable for the dataset (and study potentially redundant ones)

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE, mutual_info_regression, mutual_info_classif
from sklearn.model_selection import train_test_split
import seaborn as sns

# load data
df = pd.read_csv("../heart.csv")

# some missing values, let's drop them for now
df = df.dropna()

# change this to your own target variable
TARGET_VARIABLE = "HeartDisease"

## Handling the categorical variables

#### Type 1: Boolean categories

In [ ]:
# this just converts the value of column to 0 or 1
# factorize in pandas works too, but only one column at a time
from sklearn.preprocessing import LabelEncoder
variables = ['ExerciseAngina']
encoder = LabelEncoder()
df[variables] = df[variables].apply(encoder.fit_transform)

#### Type 2: Ordinal categories

In [ ]:
cut_mapper = {'Down': 1, 'Flat': 2,'Up': 3}
df['ST_Slope'] = df['ST_Slope'].map(cut_mapper)

#### Type 3: Nominal categories

In [ ]:
# this makes multiple columns with the variable (Separate for yes/no)
from sklearn.preprocessing import OneHotEncoder
variables = ['Sex', 'ChestPainType', 'RestingECG']

# use encoder
encoder = OneHotEncoder(sparse_output=False).set_output(transform="pandas")
one_hot_encoded = encoder.fit_transform(df[variables]).astype(int)
df = pd.concat([df,one_hot_encoded],axis=1).drop(columns=variables)

In [ ]:
df = df.drop(["Sex_F", "ChestPainType_TA", "RestingECG_LVH"], axis=1)

#### RFE - recursive feature elimination

In [ ]:
# typical X/y -split
X = df.drop(TARGET_VARIABLE, axis=1)
y = df[TARGET_VARIABLE]

# define model (linear regression, random forest, XGBoost etc.)
# technically you can use pretty much any classic ML algorithm
# model = LinearRegression()

# idea: you might want to try RFE with multuple ML algorithms
# and then cross-validate the results (which results are common in all models)

# another idea: if you plan on using e.g. XGBoost in your final model
# it's probably a good idea to use the same XGBoost here
model = RandomForestRegressor()

# create RFE, place the model and choose number of optimal variables
rfe = RFE(estimator=model, n_features_to_select=7)

# fit the RFE model with our data
rfe.fit(X, y)

# get rankings and results
rankings = rfe.ranking_
support = rfe.support_

# build a DataFrame to wrap up result for easier inspection
results_df = pd.DataFrame({
    "Feature": X.columns,
    "Ranking": rankings,
    "Selected": support
}).sort_values(by="Ranking")

# you can use these results with any other knowledge you have
# from other optimal variable selection tools, and cross-validation
results_df

,Feature,Ranking,Selected
0,Age,1,True
1,RestingBP,1,True
2,Cholesterol,1,True
4,MaxHR,1,True
7,ST_Slope,1,True
6,Oldpeak,1,True
10,ChestPainType_ASY,1,True
9,Sex_M,2,False
5,ExerciseAngina,3,False
3,FastingBS,4,False


In [ ]:

# the selection loosely follow the correlations it seems
# this might imply the data is more or less linear or something else
df.corr()[TARGET_VARIABLE].sort_values(ascending=False)

HeartDisease         1.000000
ChestPainType_ASY    0.516716
ExerciseAngina       0.494282
Oldpeak              0.403951
Sex_M                0.305445
Age                  0.282039
FastingBS            0.267291
RestingBP            0.107589
RestingECG_ST        0.102527
RestingECG_LVH       0.010670
ChestPainType_TA    -0.054790
RestingECG_Normal   -0.091580
ChestPainType_NAP   -0.212964
Cholesterol         -0.232741
Sex_F               -0.305445
MaxHR               -0.400421
ChestPainType_ATA   -0.401924
ST_Slope            -0.558771
Name: HeartDisease, dtype: float64

#### Mutual information - another alternative

In [ ]:
# fit the mutual information algorithm
# => that's why either or works
mi = mutual_info_regression(X, y)

# convert results into DataFrame
mi_results = pd.Series(mi, index=X.columns).sort_values(ascending=False)

# high value => variable is strongly connected to target variable
# low value => weak connection
# e.g. very similar to phik-matrix
mi_results

ST_Slope             0.169298
Oldpeak              0.148102
ChestPainType_ASY    0.137284
ExerciseAngina       0.131724
MaxHR                0.085199
Sex_F                0.082526
Sex_M                0.061098
Cholesterol          0.060286
Age                  0.037574
ChestPainType_ATA    0.022815
FastingBS            0.020300
RestingBP            0.008728
ChestPainType_NAP    0.003193
ChestPainType_TA     0.000000
RestingECG_LVH       0.000000
RestingECG_Normal    0.000000
RestingECG_ST        0.000000
dtype: float64